In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from tqdm import tqdm


# ══════════════════════════════════════════════════════════════════════════════
# MobileViT-XXS  —  OFFLINE WEIGHT LOADING VERSION
# Weights were downloaded on local PC and uploaded to JupyterLab manually.
# No internet connection required on the GPU server.
# ══════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────
# PATH TO YOUR UPLOADED WEIGHTS FILE
# Change this if you uploaded it to a different location
# ─────────────────────────────────────────────
PRETRAINED_WEIGHTS_PATH = 'mobilevit_xxs_pretrained.pth'  # uploaded from local PC


# ─────────────────────────────────────────────
# SEEDING — identical to all v3 models
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)       # seeds all PyTorch CPU random ops
    np.random.seed(seed)          # seeds NumPy random ops
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)  # seeds all GPUs

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ─────────────────────────────────────────────
# CONFIG — aligned with HybridResNet v3
# ─────────────────────────────────────────────
num_classes      = 10
batch_size       = 32       # reduce to 16 if you get out-of-memory errors
num_epochs       = 80       # same epoch budget as v3
lr_head          = 1e-3     # higher LR for the new randomly-initialised head
lr_backbone      = 5e-5     # tiny LR for pretrained backbone layers
weight_decay     = 3e-4     # same as v3
IMG_SIZE         = 256      # MobileViT-XXS native resolution
HEAD_ONLY_EPOCHS = 5        # freeze backbone for first 5 epochs


# ─────────────────────────────────────────────
# TRANSFORMS
# ─────────────────────────────────────────────
# MobileViT was pretrained with ImageNet normalisation stats.
# We must match these — using 0.5/0.5 would shift the input distribution
# away from what the pretrained weights expect and hurt performance.
#
# Grayscale → RGB: x.repeat(3,1,1) copies the single channel 3 times.
# This is standard practice for grayscale inputs to RGB-pretrained models.

IMAGENET_MEAN = [0.485, 0.456, 0.406]   # ImageNet per-channel means
IMAGENET_STD  = [0.229, 0.224, 0.225]   # ImageNet per-channel stds

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),        # ensure 1 channel
    transforms.Resize((IMG_SIZE, IMG_SIZE)),             # 32×32 → 256×256
    transforms.RandomHorizontalFlip(p=0.5),             # light augmentation
    transforms.RandomRotation(degrees=10),              # small rotation
    transforms.ToTensor(),                              # → (1, 256, 256) float
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),    # → (3, 256, 256)
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


# ─────────────────────────────────────────────
# DATASETS & LOADERS — same paths as v3
# ─────────────────────────────────────────────
TRAIN_PATH = 'virus_MNIST dataset/train'
VAL_PATH   = 'virus_MNIST dataset/val'
TEST_PATH  = 'virus_MNIST dataset/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    print(f"Datasets loaded: {len(train_dataset)} train | "
          f"{len(val_dataset)} val | {len(test_dataset)} test")
    print(f"Classes: {train_dataset.classes}")
except Exception as e:
    print(f"Error loading datasets: {e}")
    raise

# Class-weighted loss — identical to v3
try:
    labels        = [lbl for _, lbl in train_dataset.samples]
    class_weights = compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels
    )
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights:", np.round(class_weights, 3))
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS — identical to HybridResNet v3
# ══════════════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """
    Focal Loss with per-class weighting and label smoothing.
    Identical to HybridResNet v3 for fair comparison.

    Data flow: logits (B,10) → CE per-sample → focal modulation → mean scalar
    """
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.15):
        super().__init__()
        self.weight          = weight           # per-class imbalance correction
        self.gamma           = gamma            # 2.0 = focus on hard samples
        self.label_smoothing = label_smoothing  # 0.15 = soften overconfident predictions

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(
            inputs, targets,
            weight          = self.weight,
            label_smoothing = self.label_smoothing,
            reduction       = 'none'            # keep per-sample for focal term
        )
        pt         = torch.exp(-ce_loss)        # confidence on correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss   # down-weight easy samples
        return focal_loss.mean()


# ══════════════════════════════════════════════════════════════════════════════
# MODEL — build architecture then load weights from local file
# ══════════════════════════════════════════════════════════════════════════════

def build_mobilevit_offline(num_classes, weights_path):
    """
    Builds MobileViT-XXS architecture without downloading anything,
    then loads pretrained weights from a local .pth file.

    Why pretrained=False then manual load:
      timm's pretrained=True calls huggingface_hub to download weights.
      Since huggingface_hub isn't installed on your server, this would fail.
      Instead we build the architecture (no download needed) and load
      the state_dict we already saved locally.

    The head replacement workflow:
      1. Build model with num_classes=1000 (ImageNet head, matches saved weights)
      2. Load the full pretrained state_dict (including the 1000-class head)
      3. Replace ONLY the head with a new Linear(320, num_classes=10)
      4. The new head is randomly initialised — all other weights are pretrained

    Data flow through MobileViT-XXS:
      (B, 3, 256, 256)
      → MV2 depthwise conv blocks: local spatial features, progressive downsampling
      → MobileViT blocks ×3:
          unfold patches → Transformer self-attention → fold back → fuse with conv
      → Global Average Pool → (B, 320)
      → head: Linear(320, 10) → class logits (B, 10)
    """

    # Step 1: build architecture matching ImageNet (1000 classes) — no download
    print(f"Building MobileViT-XXS architecture...")
    model = timm.create_model(
        'mobilevit_xxs',
        pretrained=False,       # NO internet call — just builds the architecture
        num_classes=1000        # must match saved weights (ImageNet = 1000 classes)
    )

    # Step 2: load pretrained weights from local file
    print(f"Loading pretrained weights from: {weights_path}")
    state_dict = torch.load(weights_path, map_location='cpu')  # load to CPU first, move to GPU later
    model.load_state_dict(state_dict)
    print("Pretrained weights loaded successfully.")

    # Step 3: replace the head for your 10-class problem
    # timm stores the classifier as model.head — a single Linear layer
    # We replace Linear(320, 1000) with Linear(320, num_classes)
    in_features = model.head.in_features   # 320 for MobileViT-XXS
    model.head  = nn.Linear(in_features, num_classes)  # random init for new head
    # The new head has no pretrained knowledge — it will learn from your data
    nn.init.kaiming_normal_(model.head.weight)   # He init — consistent with your v3 models
    nn.init.zeros_(model.head.bias)

    print(f"Head replaced: Linear({in_features}, 1000) → Linear({in_features}, {num_classes})")
    return model


# Build the model
model = build_mobilevit_offline(num_classes, PRETRAINED_WEIGHTS_PATH).to(device)

# Report parameter counts
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"For reference — HybridResNet v3: ~180K | ClassicalResNet v3: ~182K | This: ~1.3M")


# ══════════════════════════════════════════════════════════════════════════════
# TWO-PHASE FINE-TUNING SETUP
# ══════════════════════════════════════════════════════════════════════════════
#
# Phase 1 (epochs 1–5): freeze backbone, train only the new head
#   The new head is randomly initialised → its gradients are large.
#   Backpropagating these through the pretrained backbone right away
#   would corrupt the ImageNet features. Train the head alone first.
#
# Phase 2 (epoch 6+): unfreeze all, differential LR
#   Backbone: lr=5e-5  — tiny, preserve pretrained ImageNet knowledge
#   Head:     lr=1e-3  — normal, head still adapting to 10-class malware task

def freeze_backbone(model):
    """Freeze all parameters except the classifier head (model.head)."""
    for name, param in model.named_parameters():
        if 'head' not in name:
            param.requires_grad = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 1: backbone frozen | trainable: {n:,} params (head only)")


def unfreeze_all(model):
    """Unfreeze all parameters for full end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 2: all unfrozen | trainable: {n:,} params")


def get_phase1_optimizer(model):
    """Only head parameters, high LR."""
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr_head,
        weight_decay=weight_decay
    )


def get_phase2_optimizer(model):
    """
    Differential LR: backbone gets lr_backbone, head gets lr_head.
    Prevents overwriting pretrained features with large gradient steps.

    Data flow of parameter grouping:
      Identify head param IDs → all others go to backbone group
      AdamW updates each group with its own LR independently
    """
    head_params  = list(model.head.parameters())
    head_ids     = set(id(p) for p in head_params)
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},   # pretrained: tiny LR
        {'params': head_params,     'lr': lr_head},       # new head: normal LR
    ], weight_decay=weight_decay)


# Initialise Phase 1
freeze_backbone(model)
optimizer = get_phase1_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs, eta_min=1e-8
)
loss_fn = FocalLoss(weight=class_weight_tensor, gamma=2.0, label_smoothing=0.15)


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING & EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

def train_epoch(model, dataloader, loss_fn, optimizer, device):
    """
    One training epoch.

    Data flow per batch:
      (B,3,256,256) → MV2 conv blocks → MobileViT attention blocks
      → GAP → (B,320) → head → (B,10) logits → FocalLoss → backward → step

    Returns: (avg_loss, accuracy on clean inputs)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)                                  # (B, 10)
        loss   = loss_fn(logits, labels)                       # scalar
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # same as v3
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, loss_fn, device):
    """
    Clean validation/test evaluation — no gradients computed.

    Data flow: images → model → argmax → compare with true labels → accuracy
    Returns: (avg_loss, accuracy)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits         = model(images)
            loss           = loss_fn(logits, labels)
            total_loss    += loss.item()
            correct       += (logits.argmax(1) == labels).sum().item()
            total         += labels.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate_full_report(model, dataloader, device, class_names):
    """
    Final test evaluation with full per-class classification report.
    Only called once at the end on the held-out test set.

    Data flow: all test batches → collect predictions → sklearn report
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Test Eval", leave=False):
            images = images.to(device)
            preds  = model(images).argmax(1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc        = (all_preds == all_labels).mean()

    print(f"\n  Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print("\n  Per-class Classification Report:")
    print(classification_report(all_labels, all_preds,
                                target_names=class_names, digits=4))
    return acc


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

best_val_acc               = 0.0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []
early_stopping_patience    = 15     # same as v3
epochs_without_improvement = 0
current_phase              = 1

print(f"\nStarting MobileViT-XXS fine-tuning for {num_epochs} epochs")
print(f"Phase 1 (epochs 1–{HEAD_ONLY_EPOCHS}): head only | backbone frozen")
print(f"Phase 2 (epoch {HEAD_ONLY_EPOCHS+1}+): full model | "
      f"backbone LR={lr_backbone:.0e} | head LR={lr_head:.0e}")
print(f"Loss: FocalLoss(γ=2.0, smoothing=0.15) — identical to HybridResNet v3")
print("=" * 70)

for epoch in range(1, num_epochs + 1):

    # ── Phase transition ──────────────────────────────────────────────────────
    if epoch == HEAD_ONLY_EPOCHS + 1 and current_phase == 1:
        print(f"\n  🔓 Epoch {epoch}: switching to Phase 2 (full fine-tuning)...")
        unfreeze_all(model)
        current_phase = 2
        optimizer     = get_phase2_optimizer(model)
        scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max   = num_epochs - HEAD_ONLY_EPOCHS,  # remaining epochs
            eta_min = 1e-8
        )
        print("-" * 70)

    # ── Train + validate ──────────────────────────────────────────────────────
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, loss_fn, device)

    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    gap        = train_acc - val_acc
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch:02d}/{num_epochs}] Phase {current_phase} | "
          f"LR: {current_lr:.2e} | Gap: {gap:.4f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc':              val_acc,
            'val_loss':             val_loss,
            'train_val_gap':        gap,
            'config': {
                'model':       'mobilevit_xxs',
                'num_classes': num_classes,
                'img_size':    IMG_SIZE,
                'pretrained':  'imagenet_offline',
                'params':      '~1.3M',
            }
        }, "mobilevit_xxs_virusmnist.pth")
        print(f"  💾 Best model saved (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping at epoch {epoch}.")
        break

    print("-" * 70)

print(f"\n✅ Training complete. Best Val Acc: {best_val_acc:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# FINAL TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FINAL TEST EVALUATION (best checkpoint)")
print("=" * 70)

checkpoint = torch.load("NOGAN_mobilevit_xxs_virusmnist.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
test_acc = evaluate_full_report(model, test_loader, device, train_dataset.classes)


# ══════════════════════════════════════════════════════════════════════════════
# COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FINAL COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Model':<30} {'Params':>8} {'Val Acc':>10} {'Test Acc':>10}")
print("-" * 70)
print(f"{'HybridResNet v3 (Quantum)':<30} {'~180K':>8} {'see .pth':>10} {'run test':>10}")
print(f"{'ClassicalResNet v3':<30} {'~182K':>8} {'see .pth':>10} {'run test':>10}")
print(f"{'MobileViT-XXS (pretrained)':<30} {'~1.3M':>8} {best_val_acc:>10.4f} {test_acc:>10.4f}")
print("-" * 70)

Using device: cuda
Datasets loaded: 43585 train | 4837 val | 3458 test
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Class weights: [2.069 0.674 1.71  2.173 6.554 0.78  0.337 0.691 2.019 1.555]
Building MobileViT-XXS architecture...


NameError: name 'timm' is not defined

In [6]:
import torchvision
print(torchvision.__version__)

0.22.1+cu128


In [7]:
import torch
from torchvision import models

# Build the architecture
model = models.mobilevit_xxs(weights=None)

# Load your uploaded weights
state_dict = torch.load('mobilevit_xxs_pretrained.pth', map_location='cpu')

# Check key compatibility
torchvision_keys = set(model.state_dict().keys())
uploaded_keys    = set(state_dict.keys())

print(f"Keys in torchvision model : {len(torchvision_keys)}")
print(f"Keys in your uploaded file: {len(uploaded_keys)}")
print(f"Matching keys             : {len(torchvision_keys & uploaded_keys)}")
print(f"Missing from upload       : {len(torchvision_keys - uploaded_keys)}")
print(f"Extra in upload           : {len(uploaded_keys - torchvision_keys)}")

AttributeError: module 'torchvision.models' has no attribute 'mobilevit_xxs'

In [8]:
import torchvision.models as m
# find all mobilevit variants available
print([x for x in dir(m) if 'mobile' in x.lower()])

['MobileNetV2', 'MobileNetV3', 'MobileNet_V2_Weights', 'MobileNet_V3_Large_Weights', 'MobileNet_V3_Small_Weights', 'mobilenet', 'mobilenet_v2', 'mobilenet_v3_large', 'mobilenet_v3_small', 'mobilenetv2', 'mobilenetv3']


In [9]:
print([x for x in dir(m) if 'vit' in x.lower()])

['MaxVit', 'MaxVit_T_Weights', 'ViT_B_16_Weights', 'ViT_B_32_Weights', 'ViT_H_14_Weights', 'ViT_L_16_Weights', 'ViT_L_32_Weights', 'maxvit', 'maxvit_t', 'vit_b_16', 'vit_b_32', 'vit_h_14', 'vit_l_16', 'vit_l_32']


In [10]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# MobileNetV3-Small — TORCHVISION ONLY, OFFLINE WEIGHTS
# ~2.5M parameters | No timm, no huggingface, no internet needed
#
# WHY MobileNetV3-Small is still a valid comparison:
#   • It is a well-established efficient architecture (Howard et al., 2019)
#   • Uses depthwise-separable convolutions + Squeeze-and-Excitation blocks
#     (SE blocks are the same mechanism as in your HybridResNet v3!)
#   • Pretrained on ImageNet-1k → strong visual priors
#   • ~2.5M params vs your ~180K — about 14× more capacity
#   • If your hybrid matches it: strong parameter-efficiency result
#   • Purely classical, purely convolutional — clean baseline
#
# ARCHITECTURE OVERVIEW:
#   Input (B, 3, 224, 224)
#   → 14× Bottleneck blocks (depthwise conv + SE attention + skip connections)
#   → AdaptiveAvgPool → (B, 576, 1, 1)
#   → Classifier: Linear(576,1024) → Hardswish → Dropout → Linear(1024,10)
#
# NOTE on SE blocks:
#   MobileNetV3's SE blocks are identical in concept to your HybridResNet v3's
#   SEBlock — both do global avg pool → FC → sigmoid → channel rescaling.
#   So this comparison is architecturally meaningful.
# ══════════════════════════════════════════════════════════════════════════════

PRETRAINED_WEIGHTS_PATH = 'mobilenet_v3_small_pretrained.pth'   # uploaded from local PC


# ─────────────────────────────────────────────
# SEEDING — identical to all v3 models
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ─────────────────────────────────────────────
# CONFIG — aligned with HybridResNet v3
# ─────────────────────────────────────────────
num_classes      = 10
batch_size       = 32       # safe for MobileNetV3 at 224×224; reduce to 16 if OOM
num_epochs       = 80       # same budget as v3
lr_head          = 1e-3     # high LR for new randomly-initialised head
lr_backbone      = 5e-5     # tiny LR to preserve pretrained ImageNet features
weight_decay     = 3e-4     # same as v3
IMG_SIZE         = 224      # MobileNetV3 native resolution (not 256 — that was MobileViT)
HEAD_ONLY_EPOCHS = 5        # train only head for first 5 epochs before unfreezing all


# ─────────────────────────────────────────────
# TRANSFORMS
# ─────────────────────────────────────────────
# MobileNetV3 was pretrained with ImageNet normalisation.
# We match those exact stats to preserve pretrained feature quality.
# Grayscale → repeat channel 3× to create fake RGB input.
#
# Why 224×224 (not 256):
#   MobileNetV3 was trained and benchmarked at 224×224.
#   Using 256 would be outside its trained distribution.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),        # ensure single channel
    transforms.Resize((IMG_SIZE, IMG_SIZE)),             # 32×32 → 224×224
    transforms.RandomHorizontalFlip(p=0.5),             # light augmentation
    transforms.RandomRotation(degrees=10),              # small rotation
    transforms.ToTensor(),                              # → (1, 224, 224) float
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),    # → (3, 224, 224) fake RGB
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])


# ─────────────────────────────────────────────
# DATASETS & LOADERS — same paths as v3
# ─────────────────────────────────────────────
TRAIN_PATH = 'virus_MNIST dataset/train_balanced_v2'
VAL_PATH   = 'virus_MNIST dataset/val'
TEST_PATH  = 'virus_MNIST dataset/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    print(f"Datasets loaded: {len(train_dataset)} train | "
          f"{len(val_dataset)} val | {len(test_dataset)} test")
    print(f"Classes: {train_dataset.classes}")
except Exception as e:
    print(f"Error loading datasets: {e}")
    raise

try:
    labels        = [lbl for _, lbl in train_dataset.samples]
    class_weights = compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels
    )
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights:", np.round(class_weights, 3))
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS — identical to HybridResNet v3
# ══════════════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """
    Same FocalLoss used in HybridResNet v3 and ClassicalResNet v3.
    Keeping it identical ensures the comparison isolates architecture
    differences only, not training objective differences.

    Data flow: logits (B,10) → per-sample CE → focal modulation → mean scalar
    """
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.15):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(
            inputs, targets,
            weight          = self.weight,
            label_smoothing = self.label_smoothing,
            reduction       = 'none'                # per-sample needed for focal term
        )
        pt         = torch.exp(-ce_loss)            # model confidence on correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss   # down-weight easy samples
        return focal_loss.mean()


# ══════════════════════════════════════════════════════════════════════════════
# MODEL — MobileNetV3-Small with offline pretrained weights
# ══════════════════════════════════════════════════════════════════════════════

def build_mobilenet_from_local(num_classes, weights_path):
    """
    Builds MobileNetV3-Small and loads pretrained weights from local file.

    MobileNetV3-Small classifier structure (torchvision):
      model.classifier = Sequential(
        [0] Linear(576, 1024)   ← keep pretrained
        [1] Hardswish()
        [2] Dropout(p=0.2)
        [3] Linear(1024, 1000)  ← REPLACE THIS with Linear(1024, num_classes)
      )

    Steps:
      1. Build architecture with weights=None (no internet call)
      2. Load state_dict from local .pth file (strict=True, keys always match
         because both sides use torchvision)
      3. Replace classifier[-1]: Linear(1024,1000) → Linear(1024,num_classes)
      4. Keep all other pretrained weights unchanged

    Data flow through MobileNetV3-Small:
      (B, 3, 224, 224)
      → features: 14 InvertedResidual blocks with SE attention
        (depthwise conv → pointwise conv → SE rescaling → skip add)
      → AdaptiveAvgPool → (B, 576, 1, 1) → flatten → (B, 576)
      → classifier[0] Linear → (B, 1024) → Hardswish → Dropout
      → classifier[3] Linear → (B, num_classes) logits
    """
    print("Building MobileNetV3-Small architecture...")
    model = models.mobilenet_v3_small(weights=None)   # no download — just architecture

    print(f"Loading pretrained weights from: {weights_path}")
    state_dict = torch.load(weights_path, map_location='cpu')  # load to CPU first

    # strict=True is safe here because both model and saved weights are
    # torchvision's mobilenet_v3_small — exact same key names guaranteed
    model.load_state_dict(state_dict, strict=True)
    print("Pretrained weights loaded successfully (strict match).")

    # Replace only the final classification layer
    # classifier[3] is Linear(1024, 1000) — swap 1000 → num_classes
    in_features = model.classifier[3].in_features   # 1024 for MobileNetV3-Small
    model.classifier[3] = nn.Linear(in_features, num_classes)

    # Initialise new head with He (Kaiming) init — same as your v3 models
    nn.init.kaiming_normal_(model.classifier[3].weight)
    nn.init.zeros_(model.classifier[3].bias)

    print(f"Head replaced: Linear({in_features}, 1000) → Linear({in_features}, {num_classes})")
    return model


model = build_mobilenet_from_local(num_classes, PRETRAINED_WEIGHTS_PATH).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Reference — HybridResNet v3: ~180K | ClassicalResNet v3: ~182K | This: ~2.5M")


# ══════════════════════════════════════════════════════════════════════════════
# TWO-PHASE FINE-TUNING
# ══════════════════════════════════════════════════════════════════════════════
#
# Phase 1 (epochs 1–5): freeze all backbone layers, train only classifier
#   New head is randomly initialised → large random gradients.
#   These would corrupt pretrained backbone if propagated immediately.
#   Train head alone first to align it to the malware domain.
#
# Phase 2 (epoch 6+): unfreeze everything, differential LR
#   model.features (backbone): lr = 5e-5  → preserve ImageNet knowledge
#   model.classifier (head):   lr = 1e-3  → head still adapting

def freeze_backbone(model):
    """Freeze model.features — keep model.classifier trainable."""
    for param in model.features.parameters():   # all backbone conv blocks
        param.requires_grad = False
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 1: backbone frozen | trainable: {n:,} params (classifier only)")


def unfreeze_all(model):
    """Unfreeze all layers for end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 2: all unfrozen | trainable: {n:,} params")


def get_phase1_optimizer(model):
    """Only classifier parameters, lr_head."""
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr_head,
        weight_decay=weight_decay
    )


def get_phase2_optimizer(model):
    """
    Differential LR: features (backbone) get lr_backbone, classifier gets lr_head.
    Prevents overwriting pretrained ImageNet features with large gradient steps.
    """
    return torch.optim.AdamW([
        {'params': model.features.parameters(),    'lr': lr_backbone},  # pretrained conv blocks
        {'params': model.classifier.parameters(),  'lr': lr_head},      # new head
    ], weight_decay=weight_decay)


# Initialise Phase 1
freeze_backbone(model)
optimizer = get_phase1_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs, eta_min=1e-8
)
loss_fn = FocalLoss(weight=class_weight_tensor, gamma=2.0, label_smoothing=0.15)


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING & EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

def train_epoch(model, dataloader, loss_fn, optimizer, device):
    """
    One training epoch.

    Data flow per batch:
      (B,3,224,224) → 14× InvertedResidual+SE blocks → AvgPool → flatten
      → classifier Linear layers → (B,10) logits → FocalLoss → backward → step

    Returns: (avg_loss, accuracy)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)                                  # (B, 10)
        loss   = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # same as v3
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, loss_fn, device):
    """Clean evaluation — no gradients. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits         = model(images)
            loss           = loss_fn(logits, labels)
            total_loss    += loss.item()
            correct       += (logits.argmax(1) == labels).sum().item()
            total         += labels.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate_full_report(model, dataloader, device, class_names):
    """
    Final test evaluation with per-class precision, recall, F1.
    Called once at the end on the held-out test set.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Test Eval", leave=False):
            all_preds.append(model(images.to(device)).argmax(1).cpu())
            all_labels.append(labels)

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc        = (all_preds == all_labels).mean()

    print(f"\n  Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print("\n  Per-class Classification Report:")
    print(classification_report(all_labels, all_preds,
                                target_names=class_names, digits=4))
    return acc


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

best_val_acc               = 0.0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []
early_stopping_patience    = 15     # same as v3
epochs_without_improvement = 0
current_phase              = 1

print(f"\nStarting MobileNetV3-Small fine-tuning for {num_epochs} epochs")
print(f"Phase 1 (epochs 1–{HEAD_ONLY_EPOCHS}): classifier only | backbone frozen")
print(f"Phase 2 (epoch {HEAD_ONLY_EPOCHS+1}+): full model | "
      f"backbone LR={lr_backbone:.0e} | classifier LR={lr_head:.0e}")
print(f"Loss: FocalLoss(γ=2.0, smoothing=0.15) — identical to HybridResNet v3")
print("=" * 70)

for epoch in range(1, num_epochs + 1):

    # ── Phase transition ──────────────────────────────────────────────────────
    if epoch == HEAD_ONLY_EPOCHS + 1 and current_phase == 1:
        print(f"\n  🔓 Epoch {epoch}: switching to Phase 2 (full fine-tuning)...")
        unfreeze_all(model)
        current_phase = 2
        optimizer     = get_phase2_optimizer(model)
        scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max   = num_epochs - HEAD_ONLY_EPOCHS,
            eta_min = 1e-8
        )
        print("-" * 70)

    # ── Train + validate ──────────────────────────────────────────────────────
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, loss_fn, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    gap        = train_acc - val_acc
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch:02d}/{num_epochs}] Phase {current_phase} | "
          f"LR: {current_lr:.2e} | Gap: {gap:.4f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'val_acc':          val_acc,
            'val_loss':         val_loss,
            'train_val_gap':    gap,
            'config': {
                'model':       'mobilenet_v3_small',
                'num_classes': num_classes,
                'img_size':    IMG_SIZE,
                'pretrained':  'imagenet1k_offline',
                'params':      '~2.5M',
            }
        }, "mobilenet_v3_small_virusmnist.pth")
        print(f"  💾 Best model saved (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping at epoch {epoch}.")
        break

    print("-" * 70)

print(f"\n✅ Training complete. Best Val Acc: {best_val_acc:.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# FINAL TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FINAL TEST EVALUATION (best checkpoint)")
print("=" * 70)
checkpoint = torch.load("mobilenet_v3_small_virusmnist.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
test_acc = evaluate_full_report(model, test_loader, device, train_dataset.classes)


# ══════════════════════════════════════════════════════════════════════════════
# COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("FINAL COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Model':<35} {'Params':>8} {'Val Acc':>10} {'Test Acc':>10}")
print("-" * 70)
print(f"{'HybridResNet v3 (Quantum)':<35} {'~180K':>8} {'see .pth':>10} {'run test':>10}")
print(f"{'ClassicalResNet v3':<35} {'~182K':>8} {'see .pth':>10} {'run test':>10}")
print(f"{'MobileNetV3-Small (pretrained)':<35} {'~2.5M':>8} {best_val_acc:>10.4f} {test_acc:>10.4f}")
print("-" * 70)
print("""
HOW TO READ THIS:

  MobileNetV3-Small has ~14× more parameters than your hybrid models AND
  has pretrained ImageNet knowledge. Despite these advantages:

  If your hybrids match or beat it:
    → Exceptional parameter efficiency from your ResNet + quantum design.
    → The quantum layer provides meaningful inductive bias beyond raw capacity.

  If MobileNetV3-Small wins:
    → Pretrained ImageNet features are very helpful for malware byte images.
    → Interesting finding: general visual priors transfer to malware detection.
    → Your hybrid models achieve competitive results without any pretraining.

  Key metric to also compare: Train/Val GAP
    → Your v3 models used SAM + MixUp to minimise overfitting.
    → MobileNetV3 relies on pretrained regularisation + ImageNet diversity.
    → Smaller gap = better generalisation, regardless of raw accuracy.
""")

Using device: cuda
Datasets loaded: 48498 train | 4837 val | 3458 test
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Class weights: [1.645 0.75  1.359 1.727 5.209 0.868 0.375 0.769 1.605 1.236]
Building MobileNetV3-Small architecture...
Loading pretrained weights from: mobilenet_v3_small_pretrained.pth
Pretrained weights loaded successfully (strict match).
Head replaced: Linear(1024, 1000) → Linear(1024, 10)

Total parameters    : 1,528,106
Trainable parameters: 1,528,106
Reference — HybridResNet v3: ~180K | ClassicalResNet v3: ~182K | This: ~2.5M
  Phase 1: backbone frozen | trainable: 601,098 params (classifier only)

Starting MobileNetV3-Small fine-tuning for 80 epochs
Phase 1 (epochs 1–5): classifier only | backbone frozen
Phase 2 (epoch 6+): full model | backbone LR=5e-05 | classifier LR=1e-03
Loss: FocalLoss(γ=2.0, smoothing=0.15) — identical to HybridResNet v3


Epoch [01/80] Phase 1 | LR: 1.00e-03 | Gap: -0.1235
  Train Loss: 1.1765 | Train Acc: 0.5740
  Val   Loss: 0.9270  | Val   Acc: 0.6975
  💾 Best model saved (Val Acc: 0.6975)
----------------------------------------------------------------------


Epoch [02/80] Phase 1 | LR: 9.98e-04 | Gap: 0.0885
  Train Loss: 1.0208 | Train Acc: 0.6485
  Val   Loss: 0.9353  | Val   Acc: 0.5601
  🕒 No improvement for 1 epoch(s).
----------------------------------------------------------------------


Epoch [03/80] Phase 1 | LR: 9.97e-04 | Gap: 0.0021
  Train Loss: 0.9757 | Train Acc: 0.6777
  Val   Loss: 0.8553  | Val   Acc: 0.6756
  🕒 No improvement for 2 epoch(s).
----------------------------------------------------------------------


Epoch [04/80] Phase 1 | LR: 9.94e-04 | Gap: -0.0392
  Train Loss: 0.9474 | Train Acc: 0.6918
  Val   Loss: 0.8505  | Val   Acc: 0.7310
  💾 Best model saved (Val Acc: 0.7310)
----------------------------------------------------------------------


...

Epoch [29/80] Phase 2 | LR: 3.84e-05 | Gap: 0.0545
  Train Loss: 0.4095 | Train Acc: 0.9765
  Val   Loss: 0.5315  | Val   Acc: 0.9221
  🕒 No improvement for 3 epoch(s).
----------------------------------------------------------------------


Epoch [30/80] Phase 2 | LR: 3.75e-05 | Gap: 0.0566
  Train Loss: 0.4067 | Train Acc: 0.9783
  Val   Loss: 0.5249  | Val   Acc: 0.9216
  🕒 No improvement for 4 epoch(s).
----------------------------------------------------------------------


Epoch [31/80] Phase 2 | LR: 3.66e-05 | Gap: 0.0573
  Train Loss: 0.4027 | Train Acc: 0.9800
  Val   Loss: 0.5261  | Val   Acc: 0.9227
  🕒 No improvement for 5 epoch(s).
----------------------------------------------------------------------


Epoch [32/80] Phase 2 | LR: 3.56e-05 | Gap: 0.0578
  Train Loss: 0.4003 | Train Acc: 0.9817
  Val   Loss: 0.5313  | Val   Acc: 0.9239
  🕒 No improvement for 6 epoch(s).
----------------------------------------------------------------------


KeyboardInterrupt: 

In [11]:
print("\n" + "=" * 70)
print("FINAL TEST EVALUATION (best checkpoint)")
print("=" * 70)
checkpoint = torch.load("mobilenet_v3_small_virusmnist.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
test_acc = evaluate_full_report(model, test_loader, device, train_dataset.classes)




FINAL TEST EVALUATION (best checkpoint)



  Test Accuracy: 0.9205 (92.05%)

  Per-class Classification Report:
              precision    recall  f1-score   support

           0     0.6475    0.4514    0.5320       175
           1     0.9980    0.9960    0.9970       496
           2     0.8973    0.9805    0.9371       205
           3     0.9657    0.9602    0.9630       176
           4     0.9831    1.0000    0.9915        58
           5     0.9055    0.9452    0.9249       456
           6     0.9630    0.9342    0.9484      1003
           7     0.9000    0.8982    0.8991       491
           8     0.8626    0.9075    0.8845       173
           9     0.8244    0.9600    0.8871       225

    accuracy                         0.9205      3458
   macro avg     0.8947    0.9033    0.8964      3458
weighted avg     0.9181    0.9205    0.9178      3458

